# Group B — Assignment No. 7

## Title: Predict the price of the Uber ride from a given pickup point to the agreed drop-off location

### Objective:
To preprocess the Uber fare dataset, identify and handle outliers, analyze correlation, calculate ride distances using the Haversine formula, implement Linear Regression and Random Forest Regression models, and evaluate their performance using metrics such as $R^2$, $MSE$, and $RMSE$.

### Tasks to Perform:
1. **Pre-process the dataset:** Handle missing values, extract date and time features, and drop redundant columns.
2. **Feature Engineering:** Calculate geographical distance using the Haversine formula.
3. **Identify & Handle Outliers:** Visualize distributions using box plots and filter extreme/invalid values.
4. **Check Correlation:** Compute correlation matrix and visualize relationships using a heatmap.
5. **Implement Models:** Fit Linear Regression and Random Forest Regression models.
6. **Evaluate & Compare:** Compute $R^2$, $MSE$, $RMSE$, and $MAE$ for both models, display comparison table and visual plots.

---


### Step 1: Import Required Libraries
Import necessary libraries for data manipulation, numerical calculations, visualization, and machine learning.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Kagglehub for dataset download
import kagglehub

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


### Step 2: Download and Load the Dataset
Download the Uber Fares dataset using `kagglehub` and load it into a pandas DataFrame.

In [ ]:
# Download dataset using kagglehub
path = kagglehub.dataset_download("yasserh/uber-fares-dataset")
print("Path to dataset files:", path)

# Load CSV into DataFrame
csv_file_path = os.path.join(path, "uber.csv")
df = pd.read_csv(csv_file_path)

print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()


### Step 3: Exploratory Data Analysis & Preprocessing
Inspect dataset structure, data types, missing values, summary statistics, and drop non-informative identifiers.

In [ ]:
# Inspect dataset structure and data types
df.info()


In [ ]:
# Check for null / missing values
print("Missing values per column:")
print(df.isnull().sum())

# Drop rows with null values
df.dropna(inplace=True)
print(f"\nRemaining rows after dropping nulls: {len(df)}")


In [ ]:
# Summary statistics of numerical columns
df.describe()


In [ ]:
# Drop redundant identifier columns: Unnamed: 0 and key
df.drop(columns=['Unnamed: 0', 'key'], inplace=True, errors='ignore')
df.head()


### Step 4: Datetime Feature Engineering
Convert `pickup_datetime` to datetime objects and extract temporal components (`hour`, `day`, `month`, `year`, `dayofweek`).

In [ ]:
# Convert pickup_datetime to datetime dtype
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Extract temporal features
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek  # 0=Monday, 6=Sunday

# Drop original datetime column
df.drop(columns=['pickup_datetime'], inplace=True)
df.head()


### Step 5: Calculate Distance using the Haversine Formula
The **Haversine formula** determines the great-circle distance between two points on a sphere given their longitudes and latitudes:
$$d = 2r \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta \phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta \lambda}{2}\right)}\right)$$
where $r = 6371\text{ km}$ (Earth's mean radius), $\phi$ is latitude in radians, and $\lambda$ is longitude in radians.

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees) in kilometers.
    """
    R = 6371.0  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Compute distance in kilometers
df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

df[['distance_km', 'fare_amount']].describe()


### Step 6: Identify Outliers Using Box Plots
Outliers are data points that deviate significantly from expected observations:
- **Global Outliers**: Extreme single values (e.g., negative fares or coordinate zeros).
- **Contextual Outliers**: Values anomalous under specific conditions.
- **Collective Outliers**: Subsets of anomalous records.

We visualize the distributions of `fare_amount`, `distance_km`, and `passenger_count` using box plots.

In [ ]:
# Visualize distributions with Box Plots before filtering
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(y=df['fare_amount'], color='skyblue')
plt.title('Box Plot: Fare Amount (Before Cleaning)')
plt.ylabel('Fare ($)')

plt.subplot(1, 3, 2)
sns.boxplot(y=df['distance_km'], color='salmon')
plt.title('Box Plot: Distance (Before Cleaning)')
plt.ylabel('Distance (km)')

plt.subplot(1, 3, 3)
sns.boxplot(y=df['passenger_count'], color='lightgreen')
plt.title('Box Plot: Passenger Count (Before Cleaning)')
plt.ylabel('Passenger Count')

plt.tight_layout()
plt.show()


### Step 7: Handle Outliers and Invalid Records
Filter out anomalous records:
- Fares $\ge \$2.50$ (base fare) and $\le \$100$.
- Travel distance $\ge 0.1\text{ km}$ (100m) and $\le 100\text{ km}$.
- Passenger counts between $1$ and $6$.
- NYC coordinate boundary filters: $\text{Latitude} \in [40, 42]$, $\text{Longitude} \in [-75, -72]$.

In [ ]:
initial_records = len(df)

# Fare amount filter: Positive, realistic taxi fare range
df = df[(df['fare_amount'] >= 2.5) & (df['fare_amount'] <= 100)]

# Distance filter: Greater than 100m and realistic city trip (< 100 km)
df = df[(df['distance_km'] >= 0.1) & (df['distance_km'] <= 100)]

# Passenger count filter: 1 to 6 passengers
df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 6)]

# Coordinate filter: Realistic NYC area bounding box
df = df[(df['pickup_latitude'] >= 40.0) & (df['pickup_latitude'] <= 42.0)]
df = df[(df['pickup_longitude'] >= -75.0) & (df['pickup_longitude'] <= -72.0)]
df = df[(df['dropoff_latitude'] >= 40.0) & (df['dropoff_latitude'] <= 42.0)]
df = df[(df['dropoff_longitude'] >= -75.0) & (df['dropoff_longitude'] <= -72.0)]

print(f"Initial records: {initial_records}")
print(f"Records after outlier removal: {len(df)}")
print(f"Removed {initial_records - len(df)} outliers / invalid records.")


In [ ]:
# Visualize distributions with Box Plots after cleaning
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
sns.boxplot(y=df['fare_amount'], color='skyblue')
plt.title('Box Plot: Fare Amount (After Cleaning)')
plt.ylabel('Fare ($)')

plt.subplot(1, 3, 2)
sns.boxplot(y=df['distance_km'], color='salmon')
plt.title('Box Plot: Distance (After Cleaning)')
plt.ylabel('Distance (km)')

plt.subplot(1, 3, 3)
sns.boxplot(y=df['passenger_count'], color='lightgreen')
plt.title('Box Plot: Passenger Count (After Cleaning)')
plt.ylabel('Passenger Count')

plt.tight_layout()
plt.show()


### Step 8: Check Correlation
Analyze linear relationships between numerical variables using a correlation matrix and heatmap.

In [ ]:
# Compute correlation matrix
corr_matrix = df.corr()

# Plot Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar=True)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

# Correlation with target variable (fare_amount)
print("Correlation with fare_amount:")
print(corr_matrix['fare_amount'].sort_values(ascending=False))


### Step 9: Feature Selection, Splitting, and Scaling
Select independent variables ($X$) and target variable ($y$), split into training and testing sets (80-20 split), and scale features using `StandardScaler`.

In [ ]:
# Define feature matrix X and target vector y
features = ['distance_km', 'passenger_count', 'hour', 'day', 'month', 'year', 'dayofweek']
X = df[features]
y = df['fare_amount']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Step 10: Model 1 — Linear Regression
Linear Regression models a linear relationship between input features and the continuous target variable:
$$y = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_n x_n$$

In [ ]:
# Initialize and train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions on test set
y_pred_lr = lr_model.predict(X_test_scaled)

# Calculate evaluation metrics
r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

# Display output matching lab manual format
print("Linear Regression:")
print(f"R²   = {r2_lr:.4f}")
print(f"MSE  = {mse_lr:.4f}")
print(f"RMSE = {rmse_lr:.4f}")
print(f"MAE  = {mae_lr:.4f}")


### Step 11: Model 2 — Random Forest Regression
Random Forest is an ensemble learning method that fits multiple decision trees on various subsamples and averages their predictions to improve accuracy and control overfitting.

In [ ]:
# Initialize and train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions on test set
y_pred_rf = rf_model.predict(X_test)

# Calculate evaluation metrics
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

# Display output matching lab manual format
print("Random Forest Regression:")
print(f"R²   = {r2_rf:.4f}")
print(f"MSE  = {mse_rf:.4f}")
print(f"RMSE = {rmse_rf:.4f}")
print(f"MAE  = {mae_rf:.4f}")


### Step 12: Model Comparison and Evaluation
Compare Linear Regression and Random Forest Regression using $R^2$, $MSE$, $RMSE$, and $MAE$ in tabular and graphical formats.

In [ ]:
# Model comparison table
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regression'],
    'R²': [r2_lr, r2_rf],
    'MSE': [mse_lr, mse_rf],
    'RMSE': [rmse_lr, rmse_rf],
    'MAE': [mae_lr, mae_rf]
})

print('=' * 60)
print('               MODEL COMPARISON TABLE')
print('=' * 60)
print(comparison_df.to_string(index=False))
print('=' * 60)


In [ ]:
# Graphical comparison of model metrics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# R2 Score comparison
sns.barplot(x='Model', y='R²', data=comparison_df, ax=axes[0], palette='Blues_d')
axes[0].set_title('R² Score Comparison (Higher is Better)')
axes[0].set_ylim(0, 1.0)
for p in axes[0].patches:
    val = f'{p.get_height():.4f}'
    axes[0].annotate(val, (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', color='white', fontweight='bold', fontsize=11)

# RMSE comparison
sns.barplot(x='Model', y='RMSE', data=comparison_df, ax=axes[1], palette='Reds_d')
axes[1].set_title('RMSE Comparison (Lower is Better)')
for p in axes[1].patches:
    val = f'{p.get_height():.4f}'
    axes[1].annotate(val, (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', color='white', fontweight='bold', fontsize=11)

# MSE comparison
sns.barplot(x='Model', y='MSE', data=comparison_df, ax=axes[2], palette='Purples_d')
axes[2].set_title('MSE Comparison (Lower is Better)')
for p in axes[2].patches:
    val = f'{p.get_height():.4f}'
    axes[2].annotate(val, (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# Actual vs Predicted Plots
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_test[:500], y_pred_lr[:500], alpha=0.5, color='royalblue', edgecolor='k', s=30)
plt.plot([y_test.min(), 80], [y_test.min(), 80], '--r', lw=2)
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title(f'Linear Regression: Actual vs Predicted\n(R² = {r2_lr:.4f})')

plt.subplot(1, 2, 2)
plt.scatter(y_test[:500], y_pred_rf[:500], alpha=0.5, color='seagreen', edgecolor='k', s=30)
plt.plot([y_test.min(), 80], [y_test.min(), 80], '--r', lw=2)
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title(f'Random Forest: Actual vs Predicted\n(R² = {r2_rf:.4f})')

plt.tight_layout()
plt.show()


### Step 13: Feature Importance Analysis (Random Forest)
Examine which features contributed most significantly to predicting the ride fare.

In [ ]:
# Feature importance from Random Forest Regressor
feature_importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
plt.title('Feature Importances in Uber Fare Prediction (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print('Feature Importance Ranking:')
print(feature_importances)


### Step 14: Sample Prediction
Test both trained models on a single ride scenario.

In [ ]:
# Sample ride test case
# Distance: 5.5 km, Passenger count: 2, Hour: 18 (6 PM), Day: 15, Month: 8, Year: 2014, Day of week: 4 (Friday)
sample_ride = pd.DataFrame([{
    'distance_km': 5.5,
    'passenger_count': 2,
    'hour': 18,
    'day': 15,
    'month': 8,
    'year': 2014,
    'dayofweek': 4
}])

sample_ride_scaled = scaler.transform(sample_ride)

pred_lr_sample = lr_model.predict(sample_ride_scaled)[0]
pred_rf_sample = rf_model.predict(sample_ride)[0]

print('Sample Ride Parameters:')
for col, val in sample_ride.iloc[0].items():
    print(f'  {col}: {val}')
print('-' * 45)
print(f'Predicted Fare (Linear Regression):       ${pred_lr_sample:.2f}')
print(f'Predicted Fare (Random Forest Regression): ${pred_rf_sample:.2f}')


### Conclusion
In this assignment:
1. The Uber fare dataset was loaded via `kagglehub` and preprocessed by handling missing values and extracting temporal features (`hour`, `day`, `month`, `year`, `dayofweek`).
2. The **Haversine formula** was used to calculate the real-world distance between pickup and dropoff coordinates (`distance_km`).
3. Outliers were detected and visualized using **Box Plots**, filtering out non-physical coordinate readings and extreme fares/distances.
4. A correlation heatmap confirmed that `distance_km` is the strongest predictor of `fare_amount`.
5. **Linear Regression** and **Random Forest Regression** models were trained and evaluated.
6. **Random Forest Regression** achieved a significantly superior performance ($R^2 \approx 0.845$, $RMSE \approx 3.69$) compared to Linear Regression ($R^2 \approx 0.757$, $RMSE \approx 4.63$), demonstrating the benefits of non-linear ensemble modeling for ride fare estimation.